In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

# Install libraries

```bash
conda create -n edu4 python=3.11 jupyter matplotlib
```

```bash 
! pip install -U -r requirements.txt
```

```bash
! pip install -U numpy
! pip install -U scikit-learn
```

# Update repository

In [ ]:
# ! git pull

# Add import path

In [ ]:
import gc
import os
import sys
from pathlib import Path

In [ ]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')

In [ ]:
add_library_level(level=5)

# Organize imports

In [ ]:
import multiprocessing

In [ ]:
from src.lattmc.utils import init_device, empty_cache

In [ ]:
from src.lattmc.fca.utils import *
from src.lattmc.fca.fca_utils import *

In [ ]:
from src.lattmc.tc.transcoder_analyzers import ConceptAnalysis, init_analyzer

In [ ]:
from src.lattmc.sae.nlp_sae_utils import gen_concept

In [ ]:
import logging

#### Number of CPU cores

In [ ]:
workers = multiprocessing.cpu_count()
workers

In [ ]:
SEED = 2026

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
device = init_device()
device

In [ ]:
torch.__version__

In [ ]:
np.__version__

In [ ]:
# np.set_printoptions(precision=4, suppress=True)

# Initialize Paths

In [ ]:
PATH = Path('data')
GPT2 = PATH / 'transcoders' / 'gpt2'
OWT_TOKENS_DIR = GPT2 / 'owt_tokens'
TOKENS_PATH = OWT_TOKENS_DIR / 'owt_tokens_torch.pt'
OWT_TOKENS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
! ls {GPT2}

In [ ]:
GPT2.absolute()

In [ ]:
! ls {TOKENS_PATH}

In [ ]:
! ls {str(TOKENS_PATH.absolute())}

# Load trancoders

In [ ]:
layers = list(range(12))
layers = [0, 4, 6, 8, 10, 11]
# layers = list(range(12))
# layers

In [ ]:
tr_analyzer = init_analyzer(
    layers,
    TOKENS_PATH,
    GPT2,
    model_name='gpt2-small',
    device=device,
    tr_or_sae=True,
    vector_dir=GPT2,
)

In [ ]:
empty_cache()

In [ ]:
tr_analyzer.transcoder.model.cfg.device

In [ ]:
empty_cache()

# Features Dimension

In [ ]:
tr_analyzer.fcas[0].V.shape, tr_analyzer.fcas[8].V.shape

# Analyze Tokens

In [ ]:
tr_analyzer.tokens.shape

In [ ]:
tr_analyzer.tokens[0][0]

In [ ]:
tn = to_numpy(tr_analyzer.tokens[0][0])
tn.item()

## New York City

In [ ]:
text = ' New York City'
text

In [ ]:
text_idx = 3457
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.corpus[3457]

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [1, 2, 3]

### New York City layer 0 

In [ ]:
layer = 0

In [ ]:
A_ny = dict()

In [ ]:
tr_analyzer.to_string(prompt[3])

In [ ]:
concept_an.prompt.squeeze().shape, concept_an.prompt.squeeze().dim()

In [ ]:
concept_an.prompt.tolist()

In [ ]:
concept_an.transcoder.model.tokenizer.decode(concept_an.prompt.tolist())

In [ ]:
str_tokens = concept_an.transcoder.to_str_tokens(prompt)   # if tokens is [seq]
# # or
# str_tokens = [model.to_single_str_token(int(t)) for t in tokens]
str_tokens

In [ ]:
str_tokens = [concept_an.transcoder.model.to_single_str_token(int(t)) for t in prompt]
str_tokens

In [ ]:
concept_an.transcoder.model.tokenizer.batch_decode(prompt, clean_up_tokenization_spaces=False)

In [ ]:
concept_an.prompt.shape, concept_an.prompt.dim()

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, red_val=[10.8, 26.8, 18.8], limit=100, min_vals=True, full_tokens=True,)

In [ ]:
# vs = concept_an.transcoder('New York City', layer)

In [ ]:
# vs.shape

In [ ]:
# concept_an.c_is

In [ ]:
# A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
# concept_an.c_is[layer].A.shape

In [ ]:
t_idcs

#### Common Features of Mebeddings

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
c.c.A.shape

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
concept_an.G_FG

### New York City layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, min_vals=True, full_tokens=True,)

In [ ]:
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
intersect(A_ny[0], A_ny[8]).shape

In [ ]:
u = concept_an.vs[layer]
u_12 = meet(u[1], u[2])
u_23 = meet(u[2], u[3])
u_13 = meet(u[1], u[3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs[2:3], vals[2:3], layer, with_text=True, limit=100)

In [ ]:
u = concept_an.vs[layer]
u12 = meet(u[t_idcs[0]], u[t_idcs[1]])
u23 = meet(u[t_idcs[1]], u[t_idcs[2]])
u13 = meet(u[t_idcs[0]], u[t_idcs[2]])
min_12 = tr_analyzer.min_nonzero_vals(layer, u12)
min_23 = tr_analyzer.min_nonzero_vals(layer, u23)
min_13 = tr_analyzer.min_nonzero_vals(layer, u13)

In [ ]:
vals, idcs = topK(u_12, k=u_12.shape[0])

In [ ]:
vals, idcs = topK(min_12, k=min_12.shape[0])
vals, idcs

In [ ]:
if np.any(u_12 > 0):
   _, c = concept_an.gen_and_print(idcs[2:3], vals[2:3], layer, with_text=True, limit=100)

In [ ]:
np.any(u23 > 0)

In [ ]:
vals, idcs = topNonZeros(min_23)
vals, idcs

In [ ]:
if np.any(u23 > 0):
    _, c = concept_an.gen_and_print(idcs[2:3], vals[2:3], layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
empty_cache()

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
if np.any(vals[1:] > 0):
    _, c = concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### New York City layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, red_val=[4, 4, 4], limit=100, set_val=0.2, min_vals=True, full_tokens=True,)

In [ ]:
A_ny[layer] = concept_an.c_is[layer].A

In [ ]:
np.setdiff1d(A_ny[0], intersect(A_ny[0], A_ny[layer])).shape

In [ ]:
np.setdiff1d(A_ny[layer], intersect(A_ny[0], A_ny[layer])).shape

In [ ]:
concept_an.corpus[concept_an.c_is[layer].A].shape

In [ ]:
concept_an.det_tokens[layer].shape

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
# c = concept_an.gen_and_print([2094, 10988, 10533], [0.38, 0.30, 0.23], layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:2] > 0):
    _, c = concept_an.gen_and_print(idcs[1:2], vals[1:2], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=20)

In [ ]:
vs_0 = concept_an.vs[0][t_idcs]
vs_8 = concept_an.vs[8][t_idcs]
vs_11 = concept_an.vs[11][t_idcs]

In [ ]:
t_idcs

In [ ]:
i_idx = 2

In [ ]:
vs_0.shape

In [ ]:
topNonZeros(join(vs_0[i_idx], vs_8[i_idx]))

In [ ]:
topNonZeros(join(vs_0[i_idx], vs_11[i_idx]))

In [ ]:
topNonZeros(join(vs_8[i_idx], vs_11[i_idx]))

## Richmond Hill

In [ ]:
text_idx = 117
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
# t_idcs = [6, 13, 30, 31]
# t_idcs = [6, 30, 31]
t_idcs = [30, 31]

### Richmond Hill Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Richmond Hill Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=4, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
gc.collect()

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

### Richmond Hill Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=4, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=12)

## National Park Service

In [ ]:
text_idx = 21481
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [24, 25, 26]

### National Park Service Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### National Park Service Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### National Park Service Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
# concept_an.gen_and_print([1944, 8298, 15520], [0.2, 0.2, 0.2], layer, with_text=True, limit=100)

In [ ]:
# if np.any(vals[1:2] > 0):
#     concept_an.gen_and_print(idcs[1:2], vals[1:2], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

## Rio de Janeiro

In [ ]:
text_idx = 5411
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [15, 16, 17]

### Rio de Janeiro Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=None, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 0

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

#### Common Features Analysis at Layer 0

In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Rio de Janeiro Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=None, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings at Layer 8

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

#### Common Features Analysis at Layer 8

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs[, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Rio de Janeiro Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=4, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings at Layer 11

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0): 
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:2] > 0):
    c = concept_an.gen_and_print(idcs[1:2], vals[1:2], layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=12)

#### Common Features Analysis at Layer 11

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

## Sport Teams Cleveland, Clippers, Cavaliers

In [ ]:
text_idx = 1924
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [15, 39, 94]

### Sport Teams Cleveland, Clippers, Cavaliers Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 0

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

#### Common Features Analysis at Layer 0

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs[, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Sport Teams Cleveland, Clippers, Cavaliers Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 8

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

In [ ]:
if np.any(vals[1:] > 0):
    c = concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print([16679, 10130, 20994], [12, 9, 9], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

#### Common Features Analysis at Layer 8

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs[, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Sport Teams Cleveland, Clippers, Cavaliers Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=2, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 11

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_3 = u[t_idcs[2]]
u_12 = meet(u_1, u_2)
u_23 = meet(u_2, u_3)
u_13 = meet(u_1, u_3)
u_123 = meet_all([u_1, u_2, u_3])
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
u_23_mins = tr_analyzer.min_nonzero_vals(layer, u_23)
u_13_mins = tr_analyzer.min_nonzero_vals(layer, u_13)
u_123_mins = tr_analyzer.min_nonzero_vals(layer, u_123)
topNonZeros(u_123_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[2]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
# concept_an.gen_and_print([4416, 13040, 13098], [2, 2, 2], layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
v_3 = concept_an.v_FG[layer][t_idcs[2]]
# v_meet = meet(v_1, v_2)
v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

#### Common Features Analysis at Layer 11

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_23_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs[, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_13_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_123_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

## Pets Dogs and Cats

In [ ]:
text_idx = 4042
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text

In [ ]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)

In [ ]:
concept_an.analyze_concepts()

In [ ]:
t_idcs = [2, 3]
t_idcs = [1]
# t_idcs = [2]
t_idcs = [8, 82]

### Dogs vs Cats Layer 0

In [ ]:
layer = 0

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 0

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_12 = meet(u_1, u_2)
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
topNonZeros(u_12_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v0_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v0_GF, v0_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

#### Common Features Analysis at Layer 0

In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

### Dogs vs Cats Layer 8

In [ ]:
layer = 8

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, set_val=0.2, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 8

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_12 = meet(u_1, u_2)
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
topNonZeros(u_12_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals[:10], idcs[:10]

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
# if np.any(vals[1:] > 0):
#     c = concept_an.gen_and_print(idcs[1:], vals[1:], layer, with_text=True, limit=10)

In [ ]:
v8_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v8_GF, v8_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

#### Common Features Analysis at Layer 8

In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

### Dogs vs Cats Layer 11

In [ ]:
layer = 11

In [ ]:
dets, vs = concept_an.gen_text(t_idcs, layer, limit=100, min_vals=True, full_tokens=True,)

#### Common Features of Embeddings and Layer 11

In [ ]:
u = concept_an.vs[layer]
print(f'{u.shape = }')
u_1 = u[t_idcs[0]]
u_2 = u[t_idcs[1]]
u_12 = meet(u_1, u_2)
u_12_mins = tr_analyzer.min_nonzero_vals(layer, u_12)
topNonZeros(u_12_mins)
# vals, idcs = topNonZeros(u_12_mins)
# vals, idcs
# if np.any(vals > 0):
#    _, c = concept_an.gen_and_print(idcs[1:3], vals[1:3], layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[0]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v11_GF = concept_an.v_FG[layer][t_idcs[1]]
vals, idcs = topK(v11_GF, v11_GF.shape[0])
vals, idcs

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
v_1 = concept_an.v_FG[layer][t_idcs[0]]
v_2 = concept_an.v_FG[layer][t_idcs[1]]
# v_3 = concept_an.v_FG[layer][t_idcs[2]]
v_meet = meet(v_1, v_2)
# v_meet = meet(v_1, v_3)
# v_meet = meet(v_2, v_3)
# v_meet = meet(v_meet, v_3)
vals, idcs = topK(v_meet, v_meet.shape[0])
vals, idcs

In [ ]:
topK(v_1, 10)

In [ ]:
topK(v_2, 10)

In [ ]:
topK(v_3, 10)

In [ ]:
if np.any(vals > 0):
    c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=10)

#### Common Features Analysis at Layer 11

In [ ]:
vals, idcs = topNonZeros(u_12)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
vals, idcs = topNonZeros(u_12_mins)
vals, idcs

In [ ]:
if np.any(vals > 0):
   _, c = concept_an.gen_and_print(idcs, vals, layer, with_text=True, limit=100)

In [ ]:
if np.any(vals[:3] > 0):
   _, c = concept_an.gen_and_print(idcs[:3], vals[:3], layer, with_text=True, limit=100)